In [156]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import OrdinalEncoder,LabelEncoder
from sklearn.model_selection import StratifiedKFold,cross_val_predict
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import joblib

In [157]:
df = pd.read_csv('/content/USGS.csv')

In [158]:
df.shape

(15577, 22)

In [159]:
df.head()

,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2025-12-29T15:30:29.804Z,26.0392,97.0716,10.000,4.2,mb,28.0,117.0,5.201,0.98,...,2026-04-01T22:49:04.040Z,"32 km SE of Sarupathar, India",earthquake,6.96,1.920,0.147,13.0,reviewed,us,us
1,2025-12-27T23:21:18.423Z,35.7405,88.5284,10.000,4.3,mb,25.0,121.0,6.406,0.77,...,2026-04-01T22:48:24.040Z,western Xizang,earthquake,12.04,1.934,0.133,16.0,reviewed,us,us
2,2025-12-27T12:42:49.553Z,26.7466,92.3604,41.698,4.5,mb,41.0,148.0,3.140,0.66,...,2026-04-01T22:48:20.040Z,"12 km WNW of Dhekiajuli, India",earthquake,10.60,8.117,0.091,35.0,reviewed,us,us
3,2025-12-26T16:10:19.195Z,23.3943,82.5615,10.000,4.2,mb,28.0,159.0,9.911,1.15,...,2026-04-01T22:48:16.040Z,"14 km N of Baikunthpur, India",earthquake,12.41,1.936,0.115,21.0,reviewed,us,us
4,2025-12-25T23:00:05.553Z,23.7096,70.4196,10.000,4.6,mb,61.0,75.0,13.020,0.69,...,2026-04-01T22:48:11.040Z,"27 km WNW of Rāpar, India",earthquake,7.38,1.854,0.063,74.0,reviewed,us,us


In [160]:
df.isnull().sum()

,0
time,0
latitude,0
longitude,0
depth,0
mag,0
magType,0
nst,5750
gap,3522
dmin,10798
rms,7


In [161]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15577 entries, 0 to 15576
Data columns (total 22 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   time             15577 non-null  object 
 1   latitude         15577 non-null  float64
 2   longitude        15577 non-null  float64
 3   depth            15577 non-null  float64
 4   mag              15577 non-null  float64
 5   magType          15577 non-null  object 
 6   nst              9827 non-null   float64
 7   gap              12055 non-null  float64
 8   dmin             4779 non-null   float64
 9   rms              15570 non-null  float64
 10  net              15577 non-null  object 
 11  id               15577 non-null  object 
 12  updated          15577 non-null  object 
 13  place            15577 non-null  object 
 14  type             15577 non-null  object 
 15  horizontalError  4384 non-null   float64
 16  depthError       7331 non-null   float64
 17  magError    

In [162]:
df.describe()

,latitude,longitude,depth,mag,nst,gap,dmin,rms,horizontalError,depthError,magError,magNst
count,15577.000000,15577.000000,15577.000000,15577.000000,9827.000000,12055.000000,4779.000000,15570.000000,4384.000000,7331.000000,4751.000000,14343.000000
mean,22.606830,87.640989,32.456385,4.421064,46.485906,115.828884,3.479324,0.904576,8.192536,7.648188,0.120160,19.248553
std,10.425017,8.704137,29.309782,0.485274,62.276477,53.735958,2.362922,0.269164,2.489718,8.092547,0.057226,29.849813
min,6.000000,68.000400,0.000000,3.000000,4.000000,10.000000,0.027000,0.060000,1.980000,0.000000,0.028000,1.000000
25%,10.838000,81.607000,10.000000,4.100000,15.000000,75.550000,1.727000,0.720000,6.500000,1.900000,0.080000,3.000000
50%,25.105100,92.296000,30.000000,4.400000,25.000000,107.000000,3.226000,0.900000,8.000000,5.300000,0.110000,9.000000
75%,32.397400,94.127000,35.000000,4.700000,50.000000,154.000000,4.815000,1.090000,9.600000,10.000000,0.150000,22.000000
max,36.000000,97.997400,364.900000,7.800000,724.000000,332.900000,33.178000,1.980000,23.400000,72.300000,0.539000,379.000000


Severity calc

In [105]:
# q33 = df['mag'].quantile(0.33)
# q66 = df['mag'].quantile(0.66)

In [106]:
# def severity(mag):

#   if mag >= 6.0 : return 'High'
#   elif mag >= 4.5 : return 'Medium'
#   else : return 'Low'

In [133]:
def severity(mag):
    return 'Significant' if mag >= 4.5 else 'Minor'

In [134]:
df['severity'] = df['mag'].apply(severity)

In [135]:
df['severity'].value_counts()

,count
severity,
Minor,8600
Significant,6977


#Feature Engineering

In [136]:
df['time']      = pd.to_datetime(df['time'])
df['month']     = df['time'].dt.month
df['hour']      = df['time'].dt.hour
df['year']      = df['time'].dt.year
df['log_depth'] = np.log1p(df['depth'].clip(lower=0))
df['depth_zone'] = pd.cut(
    df['depth'],
    bins=[0, 70, 300, 700],
    labels=['Shallow', 'Intermediate', 'Deep']
)

In [137]:
def get_seismic_region(row):
    lat = row['latitude']
    lon = row['longitude']
    if lat >= 28:  return 'Himalayan'
    elif lon <= 72: return 'Western'
    elif lat <= 15: return 'Southern'
    elif lon >= 90: return 'Northeast'
    else:           return 'Central'

In [138]:
df['region'] = df.apply(get_seismic_region, axis=1)

In [139]:
features = ['depth', 'log_depth', 'latitude', 'longitude',
            'month', 'hour', 'year', 'region', 'depth_zone']

In [140]:
X = df[features].copy()
#X[['region']] = encoder.fit_transform(X[['region']].astype(str))

In [142]:
encoder = OrdinalEncoder(
    handle_unknown = 'use_encoded_value',
    unknown_value=-1
)

In [143]:
X[['region', 'depth_zone']] = encoder.fit_transform(
    X[['region', 'depth_zone']].astype(str)
)

In [144]:
X = X.fillna(X.median())

In [145]:
le = LabelEncoder()

In [146]:
y = le.fit_transform(df['severity'])

SMOTE

In [147]:
model3 = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    eval_metric='mlogloss',
    random_state=42,
    verbosity=0
)

In [148]:
# sm = SMOTE(random_state=42)
# X_res, y_res = sm.fit_resample(X, y)

# print('Before SMOTE:', dict(zip(le.classes_, [sum(y==i) for i in range(3)])))
# print('After SMOTE: ', dict(zip(le.classes_, [sum(y_res==i) for i in range(3)])))

In [149]:
pipe = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('clf', model3)
])

In [150]:
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# #sw = compute_sample_weight('balanced', y)
# preds = cross_val_predict(model3, X, y, cv=cv)

In [151]:
cv    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
preds = cross_val_predict(pipe, X, y, cv=cv)

In [152]:
print(classification_report(y, preds, target_names=le.classes_))

              precision    recall  f1-score   support

       Minor       0.67      0.67      0.67      8600
 Significant       0.59      0.60      0.60      6977

    accuracy                           0.64     15577
   macro avg       0.63      0.63      0.63     15577
weighted avg       0.64      0.64      0.64     15577



In [153]:
print('Accuracy:', round(accuracy_score(y, preds), 4))

Accuracy: 0.6377


In [154]:
cm_df = pd.DataFrame(
    confusion_matrix(y, preds),
    index=le.classes_,
    columns=le.classes_
)
print(cm_df)

             Minor  Significant
Minor         5737         2863
Significant   2780         4197


In [155]:
pipe.fit(X, y)
joblib.dump(pipe, 'model3_earthquake_severity.pkl')
joblib.dump(encoder, 'model3_encoder.pkl')
joblib.dump(le, 'model3_label_encoder.pkl')
print("Model saved.")

Model saved.
